# YOLOX-Tiny: setup

Run code cells in order using your existing CUDA training kernel. This notebook checks dependencies, downloads a pinned copy of the official Apache-2.0 YOLOX source and its COCO-pretrained Tiny checkpoint, and verifies that a two-class detector can load it. No training runs here.

We use the source directly, avoiding the upstream installer’s old ONNX-simplifier dependency and optional C++ evaluation extension. Training and evaluation use our notebook-native PyTorch helpers and do not import Ultralytics.

If the dependency check reports missing packages, run the printed command in a terminal, restart the kernel, and rerun. It only lists missing support packages; keep your existing matching CUDA PyTorch/torchvision installation.

Official source and licence: https://github.com/Megvii-BaseDetection/YOLOX

After this notebook, run 08_yolox_all_data_train.ipynb. Run 09_yolox_evaluate.ipynb after training finishes.

In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

import torch

print("Python executable:", sys.executable)
print("Python version:", sys.version)

for package in ("torch", "torchvision", "numpy", "yolox"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    probe = torch.randn(64, 64, device="cuda")
    result = probe @ probe.T
    torch.cuda.synchronize()

    print("CUDA calculation passed:", torch.isfinite(result).all().item())

    del probe, result


Python executable: /home/joe/.cache/pypoetry/virtualenvs/id-capture-MVkOamB--py3.12/bin/python3.12
Python version: 3.12.13 (main, Oct  9 2026, 17:24:42) [GCC 15.2.0]
torch: 2.14.1
torchvision: 0.29.1
numpy: 2.5.3
yolox: not installed
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
CUDA calculation passed: True


In [3]:
from pathlib import Path
import importlib.util
import shlex

PROJECT = Path("/home/joe/PycharmProjects/id-capture")
SOURCE = PROJECT / "training/vendor/YOLOX"
WEIGHTS = PROJECT / "training/weights/yolox_tiny_coco.pth"

packages = {
    "loguru": "loguru", "tqdm": "tqdm", "tabulate": "tabulate",
    "psutil": "psutil", "PIL": "Pillow", "cv2": "opencv-python",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    command = [sys.executable, "-m", "pip", "install", *missing]
    print("Run in a terminal:", shlex.join(command))
    raise RuntimeError("Install missing support packages, restart the kernel, and rerun.")
assert torch.cuda.is_available(), "Select the existing CUDA training kernel."
import torchvision
from torchvision.ops import nms
nms(torch.tensor([[0., 0., 10., 10.]], device="cuda"),
    torch.tensor([0.9], device="cuda"), 0.5)
print("PASS: matching torchvision CUDA NMS is available.")

PASS: matching torchvision CUDA NMS is available.


## Download the official source and pretrained checkpoint

The next cell downloads files only when missing. It preserves the upstream LICENSE and records the pinned source revision and file hashes. The source archive remains local; existing YOLO11 checkpoints are never loaded. The upstream repository declares Apache-2.0 licensing; retain its notices and review the selected pretrained asset and dataset terms for deployment.

In [4]:
import hashlib
import io
import json
import urllib.request
import zipfile

COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"
SOURCE_URL = f"https://github.com/Megvii-BaseDetection/YOLOX/archive/{COMMIT}.zip"
CHECKPOINT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth"

def download(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    with urllib.request.urlopen(url, timeout=120) as response:
        with temporary.open("wb") as output:
            while block := response.read(1024 * 1024):
                output.write(block)
    temporary.replace(destination)

source_archive = PROJECT / f"training/vendor/YOLOX-{COMMIT}.zip"
if not source_archive.is_file():
    print("Downloading pinned official source...")
    download(SOURCE_URL, source_archive)

# Copy only files below the expected archive root; preserve licence notices.
with zipfile.ZipFile(source_archive) as archive:
    prefix = f"YOLOX-{COMMIT}/"
    for member in archive.infolist():
        if member.is_dir():
            continue
        assert member.filename.startswith(prefix)
        relative = Path(member.filename[len(prefix):])
        assert not relative.is_absolute() and ".." not in relative.parts
        destination = SOURCE / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))

assert (SOURCE / "LICENSE").is_file()
if not WEIGHTS.is_file():
    print("Downloading official COCO-pretrained Tiny checkpoint...")
    download(CHECKPOINT_URL, WEIGHTS)

provenance = {
    "source_commit": COMMIT, "source_url": SOURCE_URL,
    "source_archive_sha256": hashlib.sha256(source_archive.read_bytes()).hexdigest(),
    "checkpoint_url": CHECKPOINT_URL,
    "checkpoint_sha256": hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
}
(WEIGHTS.parent / "yolox_tiny_provenance.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))

{
  "source_commit": "6ddff4824372906469a7fae2dc3206c7aa4bbaee",
  "source_url": "https://github.com/Megvii-BaseDetection/YOLOX/archive/6ddff4824372906469a7fae2dc3206c7aa4bbaee.zip",
  "source_archive_sha256": "647bba7d16cc7e13feb5f2beca0296f86186c944f5a8fa9ccefc2629758bccba",
  "checkpoint_url": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth",
  "checkpoint_sha256": "9de513de589ac98bb92d3bca53b5af7b9acfa9b0bacb831f7999d0f7afaee8f0"
}


In [5]:
for directory in (str(PROJECT), str(SOURCE)):
    if directory not in sys.path:
        sys.path.insert(0, directory)

from training.yolox_workflow import make_model

model = make_model(WEIGHTS).cuda().eval()
with torch.inference_mode():
    output = model(torch.zeros(1, 3, 416, 416, device="cuda"))
assert output.ndim == 3 and output.shape[-1] == 7
print("PASS: YOLOX-Tiny forward output:", tuple(output.shape))
print("Two document classes; none is represented by no accepted detection.")
del model, output
torch.cuda.empty_cache()
print("Next: open 08_yolox_all_data_train.ipynb.")

Loaded pretrained tensors: 456 | New class-head tensors: 6
PASS: YOLOX-Tiny forward output: (1, 3549, 7)
Two document classes; none is represented by no accepted detection.
Next: open 08_yolox_all_data_train.ipynb.
